# Example using an ROI

In [ ]:
%matplotlib widget

### Importing utils

In [ ]:
import numpy as np
import alphashape
from helper.mask_extract import extract_mask
from matplotlib.patches import Polygon
from matplotlib.path import Path
import matplotlib.pyplot as plt
from matplotlib import transforms
from pyiconeus import open_path, Scan, Bps, Roi
from pyiconeus.utils.utils import transform_points_forward
from pyiconeus.models.Roi import RoiElements
from ipywidgets import *

# Scan loading and setup

### Loading the scan

In [ ]:
scan: Scan = open_path("D:\demoBPS\scanFile4DScan.scan")
voxels = scan.voxels
voxels.shape
norm_voxels = voxels

# Bps loading and adding to the scan

In [ ]:
bps: Bps = open_path("D:\demoBPS\scanFile4DScan.bps")
bps.data

In [ ]:
scan.bps = bps

# Utils function to display the ROI volume

In [ ]:
def _to_row_major(arr, n_cols=3):
    """
    Normalize to shape (n, n_cols) regardless of whether the source array was
    stored as (n_cols, n) or (n, n_cols). Diagnosis: roi.vertices / roi.faces
    from pyiconeus turned out to be (n, 3) already (n rows, one per
    vertex/face), NOT (3, n) as originally assumed -- so a blind `.T` was
    transposing correctly-shaped data into the wrong shape, causing vertex
    indices to be looked up on the coordinate axis (size 3) instead of the
    vertex axis (size n), hence "index 4 is out of bounds for axis 0 with
    size 3".
    """
    arr = np.asarray(arr)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D array, got shape {arr.shape}")
    if arr.shape[0] == n_cols and arr.shape[1] != n_cols:
        return arr.T
    return arr


In [ ]:
def volume(roi: RoiElements):
    """
    Mesh volume via the divergence theorem: sum the signed volume of the
    tetrahedron formed by each face and the origin.
    """
    v = _to_row_major(roi.vertices)
    f = _to_row_major(roi.faces).astype(int)
    v0, v1, v2 = v[f[:, 0]], v[f[:, 1]], v[f[:, 2]]
    signed_vols = np.einsum('ij,ij->i', v0, np.cross(v1, v2))
    return abs(signed_vols.sum()) / 6.0


In [ ]:
def slice_contour(roi: RoiElements, coord, axis=1):
        """
        Intersect the mesh with the plane {axis-th coordinate == coord} and
        return the convex hull of the intersection, as an (k, 2) array of
        points in the two remaining coordinates (in hull order, ready to
        pass straight to matplotlib's Polygon). Returns None if the plane
        misses the mesh (or touches it in fewer than 3 distinct points).
        """
        v = _to_row_major(roi.vertices)
        f = _to_row_major(roi.faces).astype(int)
        pts = []

        for tri in f:
            tri_v = v[tri]
            c = tri_v[:, axis]
            for i in range(3):
                j = (i + 1) % 3
                c0, c1 = c[i], c[j]
                if (c0 - coord) * (c1 - coord) < 0:
                    
                    t = (coord - c0) / (c1 - c0)
                    pts.append(tri_v[i] + t * (tri_v[j] - tri_v[i]))
                elif c0 == coord:
                    pts.append(tri_v[i])

        if len(pts) < 3:
            return None

        pts = np.array(pts)
        other_axes = [a for a in range(3) if a != axis]
        pts_2d = pts[:, other_axes]
        pts_2d = np.unique(np.round(pts_2d, 6), axis=0)

        if len(pts_2d) < 3:
            return None

        return pts_2d

In [ ]:
def apply_transform(vertices, matrix):
    """
    vertices: (n, 3) array, one row per vertex
    matrix:   (4, 4) homogeneous affine, (3, 3) rotation/scale only, or (3, 4) affine
    returns:  (n, 3) transformed vertices
    """
    vertices = np.asarray(vertices, dtype=float)
    matrix = np.asarray(matrix, dtype=float)
    n = vertices.shape[0]
 
    if matrix.shape == (4, 4):
        homo = np.hstack([vertices, np.ones((n, 1))])   # (n, 4)
        transformed = homo @ matrix.T                     # (n, 4)
        return transformed[:, :3] / transformed[:, 3:4]
    else:
        raise ValueError(f"Unexpected transform shape: {matrix.shape}")


# Roi loading

In [ ]:
roi: Roi = open_path("../tests/data/Cortex.bri")

In [ ]:
probe2Lab = scan.get_ProbeToLab()[0]
voxel2Probe = scan.get_VoxelToProbe()
Brain2Voxel = np.linalg.inv(voxel2Probe) @ np.linalg.inv(probe2Lab) @ scan.bps.data

In [ ]:
def describe(name, M):
    # rough "scale" of a transform matrix: the norm of its linear part
    scale = np.linalg.norm(M) * np.linalg.norm(np.linalg.inv(M))
    print(f"{name}: scale~{scale:.6g}")
    print(M)
    print()

describe("bps.data", scan.bps.data)
describe("probe2Lab", probe2Lab)
describe("voxel2Probe", voxel2Probe)
describe("BrainToVoxel", Brain2Voxel)

In [ ]:
measuredTimeVolumic = np.zeros(scan.nTime)
for i in range(scan.nTime):
    measuredTimeVolumic[i] = max(np.max(scan.measuredTimes[i*scan.sizeY:i*scan.sizeY+scan.sizeY]), measuredTimeVolumic[i])

print(measuredTimeVolumic)
print(len(measuredTimeVolumic))

In [ ]:
transformed_cache = {}


In [ ]:
from collections import deque

def convex_hull_indices(points: np.ndarray) -> np.ndarray:
    """
    Compute the convex hull of a 2-D point set.

    Parameters
    ----------
    points : (n, 2) ndarray

    Returns
    -------
    hull_idx : (m,) ndarray of int
        Indices into `points`, ordered counter-clockwise, forming the
        convex hull polygon (no repeated first/last point).
    """
    pts = np.asarray(points, dtype=float)
    n = pts.shape[0]
    if n < 3:
        return np.arange(n)

    order = np.lexsort((pts[:, 1], pts[:, 0]))
    sorted_pts = pts[order]

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for i in range(n):
        p = sorted_pts[i]
        while len(lower) >= 2 and cross(sorted_pts[lower[-2]], sorted_pts[lower[-1]], p) <= 0:
            lower.pop()
        lower.append(i)

    upper = []
    for i in range(n - 1, -1, -1):
        p = sorted_pts[i]
        while len(upper) >= 2 and cross(sorted_pts[upper[-2]], sorted_pts[upper[-1]], p) <= 0:
            upper.pop()
        upper.append(i)

    hull_local = lower[:-1] + upper[:-1]
    hull_idx = order[np.array(hull_local, dtype=int)]
    return hull_idx


def _dist(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))


def _k_nearest(points: np.ndarray, query: np.ndarray, candidate_idx: np.ndarray, k: int) -> np.ndarray:
    """Return up to k indices (subset of candidate_idx) closest to `query`."""
    if candidate_idx.size == 0:
        return candidate_idx
    if k is None or k >= candidate_idx.size:
        return candidate_idx
    d = np.linalg.norm(points[candidate_idx] - query, axis=1)
    nearest_local = np.argpartition(d, k - 1)[:k]
    return candidate_idx[nearest_local]


def concave_hull(points: np.ndarray, threshold: float = 4.0, k: int = 10):
    """
    Compute a 2-D concave hull following Park & Oh (2012), Algorithm 1.

    Parameters
    ----------
    points : (n, 2) ndarray
        Input dataset G = {p1, ..., pn}.
    threshold : float
        The 'N' threshold from the paper. Smaller N -> sharper / more
        concave hull. N > ~5 typically reproduces the convex hull.
        Valid/typical range reported in the paper is [0, 5].
    k : int or None
        Number of nearest candidate points to examine per edge when
        looking for the "nearest inner point" to dig towards. This is a
        performance optimization (the paper's own complexity analysis
        calls the exhaustive search the bottleneck: O(r*n)). Use
        k=None to search exhaustively over all remaining points exactly
        as written in the paper.

    Returns
    -------
    edges : list[tuple[int, int]]
        Edges of the resulting ConcaveList, each a pair of indices into
        `points`.
    boundary_idx : np.ndarray
        The ordered sequence of point indices tracing the concave hull
        polygon (useful for plotting / area calculations).
    """
    pts = np.asarray(points, dtype=float)
    n = pts.shape[0]
    if n < 4:
        hull_idx = convex_hull_indices(pts)
        edges = [(int(hull_idx[i]), int(hull_idx[(i + 1) % len(hull_idx)]))
                  for i in range(len(hull_idx))]
        return edges, hull_idx

    hull_idx = convex_hull_indices(pts)
    m = len(hull_idx)
    initial_edges = [(int(hull_idx[i]), int(hull_idx[(i + 1) % m])) for i in range(m)]


    boundary_set = set(int(i) for i in hull_idx)
    all_idx = np.arange(n)

    concave_list = []
    queue = deque(initial_edges)

    while queue:
        i, j = queue.popleft()
        pi, pj = pts[i], pts[j]

        candidate_idx = all_idx[~np.isin(all_idx, list(boundary_set))]
        if candidate_idx.size == 0:
            concave_list.append((i, j))
            continue

        midpoint = (pi + pj) / 2.0
        local_candidates = _k_nearest(pts, midpoint, candidate_idx, k)

        d_i = np.linalg.norm(pts[local_candidates] - pi, axis=1)
        d_j = np.linalg.norm(pts[local_candidates] - pj, axis=1)
        dd_per_candidate = np.minimum(d_i, d_j)

        best_local = int(np.argmin(dd_per_candidate))
        pk = int(local_candidates[best_local])
        dd = float(dd_per_candidate[best_local])

        eh = _dist(pi, pj)

        if dd > 0 and (eh / dd) > threshold:
            boundary_set.add(pk)
            queue.append((i, pk))
            queue.append((pk, j))
        else:
            concave_list.append((i, j))

    boundary_idx = _order_boundary(concave_list)

    return concave_list, boundary_idx


def _order_boundary(edges):
    if not edges:
        return np.array([], dtype=int)

    adjacency = {}
    for a, b in edges:
        adjacency.setdefault(a, []).append(b)
        adjacency.setdefault(b, []).append(a)

    start = edges[0][0]
    ordered = [start]
    prev = None
    current = start
    while True:
        neighbors = adjacency[current]
        nxt = neighbors[0] if neighbors[0] != prev else neighbors[1]
        if nxt == start:
            break
        ordered.append(nxt)
        prev, current = current, nxt
        if len(ordered) > len(edges) + 1:
            break
    return np.array(ordered, dtype=int)

In [ ]:
X_size, Z_size = norm_voxels.shape[0], norm_voxels.shape[2]
xx, zz = np.mgrid[0:X_size, 0:Z_size]
grid_pts = np.column_stack((xx.ravel(), zz.ravel()))

roi_mask_cache = {}
roi_hull_cache = {}
roi_contour_cache = {}

for rIdx, r in enumerate(roi.list):
    if rIdx not in transformed_cache:
        transformed_cache[rIdx] = apply_transform(r.vertices, Brain2Voxel)
    r.vertices = transformed_cache[rIdx]

    for pose in range(scan.sizeY):
        contour = slice_contour(r, pose, axis=1)
        roi_contour_cache[(rIdx, pose)] = contour
        if contour is None:
            roi_mask_cache[(rIdx, pose)] = np.zeros((X_size, Z_size), dtype=bool)
        else:
            hull = concave_hull(contour)
            _, boundary_idx = concave_hull(contour, threshold=2.1)
            hull_pts = contour[boundary_idx]
            roi_hull_cache[(rIdx, pose)] = hull_pts

            inside = Path(hull_pts).contains_points(grid_pts)
            roi_mask_cache[(rIdx, pose)] = inside.reshape(X_size, Z_size)

In [ ]:
tmp = transform_points_forward(voxel2Probe, [0, 0, 0])
x_1 = tmp[0]
z_1 = -tmp[2]
tmp = transform_points_forward(voxel2Probe, [norm_voxels.shape[0], 0, norm_voxels.shape[2]])
x_2 = tmp[0]
z_2 = -tmp[2]
extent = np.array([x_1, x_2, z_1, z_2])

# Interactive display

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(norm_voxels[:, int(scan.sizeY / 2), :, int(scan.nTime / 2), 0, 0], cmap='gray', interpolation='bilinear')
current_state = {"pose": int(scan.sizeY / 2), "roiValue": 0, "nTime": int(scan.nTime / 2)}
poly_artist = {"poly": None, "poly2": None}


def update(nTime, pose, roiValue):
    current_state["pose"], current_state["roiValue"], current_state["nTime"] = pose, roiValue, nTime
    img_slice = norm_voxels[:, pose, :, nTime, 0, 0]
    im.set_data(img_slice)
    im.set_extent([-0.5, img_slice.shape[1] - 0.5, img_slice.shape[0] - 0.5, -0.5])
    
 
    if poly_artist["poly"] is not None:
        poly_artist["poly"].remove()
        poly_artist["poly"] = None
    if poly_artist["poly2"] is not None:
            poly_artist["poly2"].remove()
            poly_artist["poly2"] = None
    selectedRoi = roi.list[roiValue]
    selectedRoi.vertices = transformed_cache[roiValue]

    contour = roi_contour_cache[(roiValue, pose)]
    if contour is not None:
        color = (selectedRoi.color.r, selectedRoi.color.g, selectedRoi.color.b, 0.4)
        hull_pts = roi_hull_cache[(roiValue, pose)]
        hull_xy = hull_pts[:, ::-1]
        poly_artist["poly2"] = Polygon(hull_xy, closed=False, fill=True,
                                        edgecolor=color, linewidth=1.5)
        ax.add_patch(poly_artist["poly2"])
    fig.canvas.draw_idle()

 
interact(
    update,
    nTime=widgets.IntSlider(value=int(scan.nTime / 2), min=0, max=scan.nTime - 1, step=1),
    pose=widgets.IntSlider(value=int(scan.sizeY / 2), min=0, max=scan.sizeY - 1, step=1),
    roiValue=widgets.Dropdown(
        options = [(roi.list[i].name, i) for i in range(len(roi.list))],
        value = 0,
        description = "ROI: ",
    )
)


In [ ]:
from matplotlib.path import Path

pose = current_state["pose"]
roiValue = current_state["roiValue"]

fig, ax = plt.subplots(1, scan.voxels.shape[1])
for i in range(scan.voxels.shape[1]):
    selectedRoi = roi.list[roiValue]
    if roiValue not in transformed_cache:
        transformed_cache[roiValue] = apply_transform(selectedRoi.vertices, Brain2Voxel)
    selectedRoi.vertices = transformed_cache[roiValue]
    title = ax[i].set_title(f"ROI volume: {volume(selectedRoi):.2f}")

    mask = roi_mask_cache[(roiValue, i)]

    data_over_time = norm_voxels[:, i, :, :, 0, 0]
    roiSignal = (data_over_time * mask[:, :, None]).sum(axis=(0, 1)) / mask.sum()


    ax[i].plot(measuredTimeVolumic, roiSignal)
    ax[i].set_xlabel('Time (s)')
    ax[i].set_ylabel('Mean Doppler')
    ax[i].set_title(f'ROI: {selectedRoi.name}, pose={i}')
fig.canvas.draw_idle()